In [1]:
!pip install -q sentence-transformers datasets lightgbm

import torch, numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util

print("GPU:", torch.cuda.is_available())

# Corpus: 5,000 labelled reviews (1 = positive, 0 = negative)
data = load_dataset("fancyzhx/yelp_polarity")
corpus = data["test"].shuffle(seed=7).select(range(5000))
texts = list(corpus["text"])
labels = np.array(list(corpus["label"]))

# Bi-encoder: turns each review into a 384-dimensional embedding
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda")
corpus_emb = embedder.encode(texts, batch_size=128, convert_to_tensor=True,
                             normalize_embeddings=True, show_progress_bar=True)
print("Embeddings:", corpus_emb.shape)

def search(query, top_k=5):
    q_emb = embedder.encode(query, convert_to_tensor=True, normalize_embeddings=True)
    hits = util.semantic_search(q_emb, corpus_emb, top_k=top_k)[0]
    return [(h["corpus_id"], round(h["score"], 3)) for h in hits]

for q in ["friendly staff and clean rooms", "rude service and long waiting time"]:
    print("\nQUERY:", q)
    for idx, score in search(q, 3):
        print(f"  {score} | label={labels[idx]} | {texts[idx][:120]}...")

GPU: True


README.md:   0%|          | 0.00/8.93k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  256MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 17.7MB            

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Embeddings: torch.Size([5000, 384])

QUERY: friendly staff and clean rooms
  0.745 | label=0 | Rude staff.   Below average rooms....
  0.696 | label=1 | Very friendly staff, store is clean and well stocked....
  0.604 | label=1 | What a very nice place. All the staff we dealt with were so very friendly. The hotel and room were nice and clean. I wou...

QUERY: rude service and long waiting time
  0.69 | label=0 | Most rude service ever! Expect eye rolling and annoyance for asking any questions. Do not support businesses that do not...
  0.612 | label=0 | Waited for 10 mins in lobby with nobody to be seated and no body helped us as we walked out then they wanted to help us....
  0.603 | label=0 | Slow service....


In [2]:
from sentence_transformers import CrossEncoder

# Stage 2: a cross-encoder reads query + review TOGETHER and scores relevance more precisely
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device="cuda")

def search_rerank(query, retrieve_k=50, top_k=10):
    candidates = search(query, retrieve_k)                 # stage 1: fast bi-encoder retrieval
    ids = [i for i, _ in candidates]
    scores = reranker.predict([(query, texts[i]) for i in ids])
    order = np.argsort(-scores)[:top_k]                    # stage 2: re-rank by cross-encoder
    return [(ids[j], float(scores[j])) for j in order]

# Evaluation: queries with a known sentiment; a result is "relevant" if its label matches
queries = {
    "friendly staff and excellent service": 1, "delicious food, will come back": 1,
    "clean, comfortable and great value": 1, "amazing experience, highly recommend": 1,
    "beautiful place with a lovely atmosphere": 1,
    "rude staff and terrible service": 0, "dirty place, never coming back": 0,
    "overpriced and disappointing": 0, "waited an hour and the food was cold": 0,
    "worst experience ever, avoid": 0,
}

def precision_at_k(results, target, k=10):
    return np.mean([labels[i] == target for i, _ in results[:k]])

def mrr(results, target):
    for rank, (i, _) in enumerate(results, start=1):
        if labels[i] == target:
            return 1 / rank
    return 0

p_bi, p_rr, m_bi, m_rr = [], [], [], []
for q, target in queries.items():
    bi = search(q, 10)
    rr = search_rerank(q)
    p_bi.append(precision_at_k(bi, target)); p_rr.append(precision_at_k(rr, target))
    m_bi.append(mrr(bi, target));            m_rr.append(mrr(rr, target))

print(f"Bi-encoder only : Precision@10 = {np.mean(p_bi):.3f} | MRR = {np.mean(m_bi):.3f}")
print(f"With re-ranking : Precision@10 = {np.mean(p_rr):.3f} | MRR = {np.mean(m_rr):.3f}")

print("\nExample - 'rude staff and terrible service':")
for i, s in search_rerank("rude staff and terrible service", top_k=3):
    print(f"  {s:.2f} | label={labels[i]} | {texts[i][:110]}...")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Bi-encoder only : Precision@10 = 0.940 | MRR = 1.000
With re-ranking : Precision@10 = 0.950 | MRR = 1.000

Example - 'rude staff and terrible service':
  6.47 | label=0 | The customer service is the worst one I have never met before.The attitude of the staff is rude,we just asked ...
  6.10 | label=0 | Awful, rude service. They told us they open at 6pm. At 6:10, the (2nd) rude bouncer said they needed a line an...
  6.01 | label=0 | This is the worst B&N I have ever been to. The staff was horrible. They were rude and very unhelpful. When I w...


In [3]:
import lightgbm as lgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score

# Same train/test split as your DistilBERT model, for a fair comparison
train = data["train"].shuffle(seed=42).select(range(4000))
test  = data["test"].shuffle(seed=42).select(range(1000))
X_tr_txt, y_tr = list(train["text"]), np.array(list(train["label"]))
X_te_txt, y_te = list(test["text"]),  np.array(list(test["label"]))

params = dict(n_estimators=400, learning_rate=0.05, num_leaves=31, random_state=42, verbose=-1)

# Model A: TF-IDF word features + LightGBM
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=2)
Xtr_tfidf, Xte_tfidf = tfidf.fit_transform(X_tr_txt), tfidf.transform(X_te_txt)
lgb_tfidf = lgb.LGBMClassifier(**params).fit(Xtr_tfidf, y_tr)
p_a = lgb_tfidf.predict(Xte_tfidf)

# Model B: sentence embeddings (from Step 1) + LightGBM
Xtr_emb = embedder.encode(X_tr_txt, batch_size=128, normalize_embeddings=True)
Xte_emb = embedder.encode(X_te_txt, batch_size=128, normalize_embeddings=True)
lgb_emb = lgb.LGBMClassifier(**params).fit(Xtr_emb, y_tr)
p_b = lgb_emb.predict(Xte_emb)

print("Model comparison on the same 1,000 test reviews:")
print(f"  LightGBM + TF-IDF      : accuracy {accuracy_score(y_te, p_a):.3f} | F1 {f1_score(y_te, p_a):.3f}")
print(f"  LightGBM + embeddings  : accuracy {accuracy_score(y_te, p_b):.3f} | F1 {f1_score(y_te, p_b):.3f}")
print(f"  Fine-tuned DistilBERT  : accuracy 0.913 | F1 0.905   (your PyTorch model)")

# Which words drive the TF-IDF model?
imp = lgb_tfidf.feature_importances_
vocab = np.array(tfidf.get_feature_names_out())
print("\nTop 15 most important words:", list(vocab[np.argsort(-imp)[:15]]))

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Model comparison on the same 1,000 test reviews:
  LightGBM + TF-IDF      : accuracy 0.889 | F1 0.886
  LightGBM + embeddings  : accuracy 0.866 | F1 0.863
  Fine-tuned DistilBERT  : accuracy 0.913 | F1 0.905   (your PyTorch model)

Top 15 most important words: ['not', 'and', 'great', 'good', 'delicious', 'excellent', 'amazing', 'awesome', 'best', 'no', 'love', 'worst', 'bad', 'don', 'to be']


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [4]:
!pip install -q fastapi httpx
from transformers import pipeline

# Small open-source instruction-tuned LLM (runs free on the Colab GPU)
llm = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device=0)

def rag_answer(query, k=5):
    hits = search_rerank(query, top_k=k)                      # retrieve + re-rank (Steps 1-2)
    context = "\n".join(f"- {texts[i][:300]}" for i, _ in hits)
    messages = [
        {"role": "system", "content": "You summarise customer reviews. Use only the reviews given. Be concise."},
        {"role": "user", "content": f"Question: {query}\n\nReviews:\n{context}\n\n"
                                    "Summarise in 2-3 sentences what these customers say."},
    ]
    out = llm(messages, max_new_tokens=120, do_sample=False)
    summary = out[0]["generated_text"][-1]["content"].strip()
    return summary, [{"review": texts[i][:200], "score": round(s, 2), "label": int(labels[i])} for i, s in hits]

# FastAPI service with /search
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel

app = FastAPI(title="Semantic Review Search + RAG")

class Query(BaseModel):
    query: str
    top_k: int = 5

@app.post("/search")
def search_endpoint(q: Query):
    summary, results = rag_answer(q.query, q.top_k)
    return {"query": q.query, "summary": summary, "results": results}

client = TestClient(app)
resp = client.post("/search", json={"query": "how is the customer service?", "top_k": 5}).json()
print("SUMMARY:", resp["summary"])
for r in resp["results"]:
    print(f"  {r['score']} | label={r['label']} | {r['review'][:100]}...")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

SUMMARY: The customer service is the worst experience they've ever had. They were treated rudely and refused to provide any information about their order or product. The staff's behavior was so unprofessional that it left them feeling frustrated and disappointed.
  2.85 | label=0 | The customer service is the worst one I have never met before.The attitude of the staff is rude,we j...
  0.62 | label=0 | The customer service is absolutely terrible.  I have been there 3 times now, giving it a fair chance...
  -2.0 | label=0 | Don't go to this store!  The service is always poor: the people working there are unfriendly, don't ...
  -2.34 | label=0 | Absolutely terrible customer service at this location. The front end manager has a horrible demeanor...
  -2.6 | label=1 | Awesome customer service and this place was unbelievable. Everything was safe and everything went sm...


In [5]:
readme = """# Semantic Review Search, Re-Ranking & RAG

End-to-end search system over 5,000 customer reviews: semantic retrieval, cross-encoder re-ranking,
gradient-boosting baselines and an LLM summary (RAG), served via FastAPI.

## Pipeline
1. **Retrieval:** sentence embeddings (all-MiniLM-L6-v2, 384-d), cosine similarity search
2. **Re-ranking:** cross-encoder (ms-marco-MiniLM-L-6-v2) re-scores the top 50 candidates
3. **Evaluation:** Precision@10 and MRR on sentiment-labelled queries
4. **Gradient boosting:** LightGBM on TF-IDF and on embeddings, compared with fine-tuned DistilBERT
5. **RAG:** Qwen2.5-0.5B-Instruct summarises the top-ranked reviews
6. **API:** FastAPI `/search` endpoint returning the summary and ranked results

## Results
| Search | Precision@10 | MRR |
|---|---|---|
| Bi-encoder only | 0.940 | 1.000 |
| + Cross-encoder re-ranking | 0.950 | 1.000 |

| Classifier (same 1,000 test reviews) | Accuracy | F1 |
|---|---|---|
| LightGBM + TF-IDF | 0.889 | 0.886 |
| LightGBM + embeddings | 0.866 | 0.863 |
| Fine-tuned DistilBERT (PyTorch) | 0.913 | 0.905 |

## Learnings
- Embeddings capture topic more than polarity ("friendly staff" can retrieve "rude staff").
- Re-ranking improves precision; MRR was already at its ceiling.
- LightGBM + TF-IDF is a strong, cheap and interpretable baseline.
- Small evaluation set (10 queries): results are directional, not conclusive.

Related project: [Review Sentiment Classifier - PyTorch + FastAPI](https://github.com/suneetaramanathan/Review-sentiment-pytorch-fastapi)
"""
reqs = "torch\nsentence-transformers\ntransformers\ndatasets\nlightgbm\nscikit-learn\nfastapi\nuvicorn\n"
with open("README.md", "w") as f: f.write(readme)
with open("requirements.txt", "w") as f: f.write(reqs)

from google.colab import files
files.download("README.md")
files.download("requirements.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>